# Mistralethon — run the whole website on Colab

One notebook = the entire app: **React frontend + FastAPI backend + Sentinel-Loop pipeline + fine-tuned Timestral**, exposed to the internet through a cloudflared tunnel.

Run the cells top to bottom with a **GPU runtime** (Runtime → Change runtime type → T4/A100) if you want to use `finetune-timestral`; CPU-only works for everything else.

- Data, artifacts and secrets persist on **Google Drive** (`My Drive/Colab Notebooks/Mistralethon/`).
- The public URL changes every session (free Colab disconnects after ~90 min idle / 12 h max) — keep this tab open.
- Optional: set `SENTINEL_API_KEY` in the Drive `.env` to require an `X-API-Key` header on mutating endpoints.

In [ ]:
# 1) Mount Drive, load secrets, point the pipeline at Drive-persistent dirs
from google.colab import drive
import os, shutil, pathlib

drive.mount('/content/drive')
DRIVE = '/content/drive/My Drive/Colab Notebooks/Mistralethon'
ENV_FILE = pathlib.Path(DRIVE) / '.env'
for line in ENV_FILE.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, _, v = line.partition('=')
        os.environ[k.strip()] = v.strip()

# persistent data + artifacts (survive runtime restarts)
os.environ['SENTINEL_DATA_DIR'] = f'{DRIVE}/data'
os.environ['SENTINEL_ARTIFACTS_DIR'] = f'{DRIVE}/data/artifacts'
pathlib.Path(os.environ['SENTINEL_DATA_DIR']).mkdir(parents=True, exist_ok=True)

# work on a fast local copy of the repo (Drive FS is slow for pip/npm)
REPO = '/content/Mistraton'
if not pathlib.Path(REPO).exists():
    shutil.copytree(DRIVE, REPO, ignore=shutil.ignore_patterns('.git', '.venv', 'node_modules', '__pycache__', '.pytest_cache'))
os.chdir(f'{REPO}/backend')
print('repo:', REPO)
print('data:', os.environ['SENTINEL_DATA_DIR'])
print('mistral key set:', bool(os.environ.get('MISTRAL_API_KEY')))

In [ ]:
# 2) Backend deps (FastAPI, uvicorn, mistralai, ...)
!pip install -q -r requirements.txt
# GPU fine-tuning deps — only needed for the 'finetune-timestral' pipeline stage
!pip install -q -r requirements-colab.txt

In [ ]:
# 3) Build the React frontend -> frontend/dist (served by the same FastAPI app)
import shutil, subprocess, sys
if shutil.which('node') is None:
    subprocess.run('apt-get update -qq && apt-get install -y -qq nodejs npm', shell=True, check=True)
print('node', subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip())
os.chdir(f'{REPO}/frontend')
subprocess.run(['npm', 'install', '--no-audit', '--no-fund'], check=True)
subprocess.run(['npm', 'run', 'build'], check=True)
os.environ['SENTINEL_DIST_DIR'] = f'{REPO}/frontend/dist'
print('dist ready:', os.environ['SENTINEL_DIST_DIR'])

In [ ]:
# 4) Start the website backend on 0.0.0.0:8000 (background thread, cell returns immediately)
import sys, threading, time, uvicorn
sys.path.insert(0, f'{REPO}/backend')
import colab_app

app = colab_app.build_app()
threading.Thread(target=uvicorn.run, args=(app,),
                 kwargs={'host': '0.0.0.0', 'port': 8000, 'log_level': 'warning'}, daemon=True).start()
time.sleep(2)
print('server up on http://localhost:8000')

In [ ]:
# 5) Expose it: cloudflared quick tunnel (no account needed) -> public https URL
import subprocess, threading, re, urllib.request

if not pathlib.Path('/content/cloudflared').exists():
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/content/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)

PUBLIC_URL = None
def _read(proc):
    global PUBLIC_URL
    for line in iter(proc.stdout.readline, ''):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            PUBLIC_URL = m.group(0)
            print('PUBLIC URL:', PUBLIC_URL)
            break

proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
threading.Thread(target=_read, args=(proc,), daemon=True).start()
for _ in range(30):
    if PUBLIC_URL: break
    time.sleep(1)
assert PUBLIC_URL, 'tunnel did not come up; check cell output'
print('open this in a browser:', PUBLIC_URL)

In [ ]:
# 6) Sanity check through the tunnel
print(urllib.request.urlopen(PUBLIC_URL + '/api/health').read().decode())
print(urllib.request.urlopen(PUBLIC_URL + '/sentinel/health').read().decode())

## Notes

- **Keep this tab open** — free Colab disconnects after ~90 min idle and at 12 h max; the tunnel URL then changes (re-run cell 5).
- **Uploads & artifacts persist** in `My Drive/Colab Notebooks/Mistralethon/data/` (`raw/`, `records.jsonl`, `labels.db`, `artifacts/`, `timestral-hf/`).
- Website flow: **Upload** an incident bundle → **Pipeline**: ingest → extract-metrics (Mistral Large) → label-windows (Shieldstral) → build-dataset → train-detection → evaluate → finetune-timestral (GPU) → **Watch** live streams → **Timestral** chat with the fine-tuned checkpoint.
- The same code runs on the VM (`backend/main.py` + Caddy) — `colab_app.py` only adds upload/pipeline/chat/static-UI routes on top of the shared `/sentinel` router.